# 12. Character encoder + more data ensemble

**What this notebook does:** combines the two changes that helped on their own, without training anything new. Notebook 11 added a character encoder (100k rows per language); notebook 10 trained the word model on 300k rows per language. They make different mistakes, so their average may beat both.

**Why the local score is valid.** The character model and the 300k model of fold k both trained without fold k, so their average is scored on fold k without a leak, as in notebook 09.

**Inputs.** The saved logits of notebook 11 (local) and of notebook 10 (unpack `more_data_results.zip` from Colab and copy its `artifacts/` folders into `model/artifacts/`).

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "eda").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "model")]
os.environ.setdefault("NER_WANDB", "off")
os.environ.setdefault("NER_DEVICE", "cpu")

import pandas as pd
from IPython.display import display

from nerkit.artifacts import combine_models, decode_split, load_logits, majority_vote, row_key, write_submission
from nerkit.config import WORK_DIR
from nerkit.data import build_cache, load_labeling_sample, load_test
from nerkit.decisions import DECODING, base_config
from nerkit.experiment import evaluation_sets, select_training
from nerkit.metrics import evaluate

FOLDS = (0, 1, 2)
DECODE = {name: DECODING[name] for name in ("merge", "drop_empty", "trim_number", "bias")}
CHAR_KEYS = {fold: base_config(fold=fold, char_hidden=32).key() for fold in FOLDS}
MORE_DATA_KEYS = {fold: base_config(fold=fold, uk_rows=300_000, ru_rows=300_000).key() for fold in FOLDS}
build_cache()
print("character runs:", CHAR_KEYS)
print("300k runs:", MORE_DATA_KEYS)

character runs: {0: '2e3c93d7d792', 1: '3e043599ca98', 2: '77004a4e4774'}
300k runs: {0: '00b6486099dd', 1: 'a503b30fda5a', 2: 'b1b13357bc6f'}


## Cross-validation: each fold scored by its own two models and their average

The held-out rows and the synthetic posts are the same for both runs (the folds are frozen), which `load_pair` checks.

In [2]:
def scores(rows, loaded, crf):
    result = evaluate(rows, decode_split(loaded, rows, crf=crf, **DECODE))
    return dict(f1=result["mix"]["f1"], precision=result["mix"]["precision"], recall=result["mix"]["recall"])


def load_pair(fold, split, rows):
    char, char_meta = load_logits(CHAR_KEYS[fold], split)
    more, more_meta = load_logits(MORE_DATA_KEYS[fold], split)
    if set(char) != set(more):
        raise ValueError(f"fold {fold}, {split}: the two runs cover different rows")
    average, average_crf = combine_models([char, more], [char_meta, more_meta], {row_key(r): r["text"] for r in rows})
    return {"characters": (char, char_meta["crf"]), "300k rows": (more, more_meta["crf"]),
            "characters + 300k average": (average, average_crf)}


cv_rows = []
for fold in FOLDS:
    _, _, _, heldout = select_training(base_config(fold=fold))
    posts = evaluation_sets(base_config(fold=fold), heldout)["posts"]
    heldout_models, posts_models = load_pair(fold, "heldout", heldout), load_pair(fold, "posts", posts)
    for name in heldout_models:
        cv_rows.append(dict(fold=fold, model=name, **scores(heldout, *heldout_models[name]),
                            posts_f1=scores(posts, *posts_models[name])["f1"]))
cv = pd.DataFrame(cv_rows)
display(cv.pivot(index="fold", columns="model", values="f1").round(3))
display(cv.groupby("model")[["f1", "precision", "recall", "posts_f1"]].agg(["mean", "std"]).round(3))

model,300k rows,characters,characters + 300k average
fold,,,
0,0.893,0.886,0.897
1,0.901,0.896,0.908
2,0.901,0.907,0.909


f1        precision        recall         \
                            mean    std      mean    std   mean    std   
model                                                                    
300k rows                  0.899  0.005     0.910  0.020  0.888  0.021   
characters                 0.896  0.011     0.904  0.015  0.888  0.010   
characters + 300k average  0.904  0.007     0.916  0.006  0.893  0.011   

                          posts_f1         
                              mean    std  
model                                      
300k rows                    0.882  0.007  
characters                   0.873  0.006  
characters + 300k average    0.890  0.006

## Test predictions of all six models

In [3]:
test_rows, labeling_rows = load_test(), load_labeling_sample()
ALL_KEYS = [*CHAR_KEYS.values(), *MORE_DATA_KEYS.values()]


def average(rows, split):
    loaded = [load_logits(key, split) for key in ALL_KEYS]
    logits, crf = combine_models([l for l, _ in loaded], [m for _, m in loaded], {row_key(r): r["text"] for r in rows})
    return decode_split(logits, rows, crf=crf, **DECODE)


def vote(rows, split):
    single = []
    for key in ALL_KEYS:
        logits, meta = load_logits(key, split)
        single.append(decode_split(logits, rows, crf=meta["crf"], **DECODE))
    return majority_vote(single, 3)


overview = []
for name, method, file in (("average of 6", average, "char_more_data_average.csv"),
                           ("vote, 3 of 6", vote, "char_more_data_vote3.csv")):
    predictions = method(test_rows, "test")
    write_submission(WORK_DIR / "submissions" / file, test_rows, predictions)
    pooled = evaluate(labeling_rows, method(labeling_rows, "labeling"))["pooled"]
    overview.append(dict(method=name, file=file, posts_with_locations=sum(bool(p) for p in predictions),
                         locations=sum(len(p) for p in predictions), labeling_f1=pooled["f1"]))
display(pd.DataFrame(overview).set_index("method").round(3))

,file,posts_with_locations,locations,labeling_f1
method,,,,
average of 6,char_more_data_average.csv,330,771,0.720
"vote, 3 of 6",char_more_data_vote3.csv,333,777,0.692


## Leaderboard results

In [4]:
LEADERBOARD = {  # public F1, copied from Kaggle
    "characters, average (notebook 11)": 0.59475,
    "300k rows, vote 2 of 3 (notebook 10)": 0.5817,
    "300k rows, average (notebook 10)": 0.57313,
    "characters + 300k, average of 6": 0.58753,
    "characters + 300k, vote 3 of 6": 0.58857,
}
display(pd.Series(LEADERBOARD, name="public_f1").to_frame())

,public_f1
"characters, average (notebook 11)",0.59475
"300k rows, vote 2 of 3 (notebook 10)",0.58170
"300k rows, average (notebook 10)",0.57313
"characters + 300k, average of 6",0.58753
"characters + 300k, vote 3 of 6",0.58857


**Conclusion.** The ensemble is the best model on every local measure, including the 26 labeled competition posts (0.720), but on the leaderboard it lands between its members (0.588 against 0.595 for the character models alone and 0.582 for the 300k models). Adding the 300k models pulls the character models down, the same pattern as the LSTM in notebook 09. The final submission stays the character model average of notebook 11 (0.59475).